# Pipeline de Ingesta: Construcción del Grafo de Conocimiento

Sistema Graph RAG sobre el canon de Sherlock Holmes.
Este notebook ejecuta el pipeline completo de ingesta:
1. Descarga de textos de Project Gutenberg
2. Separación en relatos individuales
3. Chunking consciente de la estructura
4. Extracción multipaso de entidades y relaciones
5. Entity resolution
6. Población del grafo en Neo4j

In [15]:
%load_ext autoreload
%autoreload 2
# Si ejecutas desde notebooks/, necesitas que graphrag sea importable.
# Con uv: uv sync --extra dev && pip install -e .
from graphrag.config import get_settings
from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor
from graphrag.ingestion.entity_extractor import EntityExtractor

settings = get_settings()
print(f"Proyecto GCP: {settings.google_cloud_project}")
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Chunk size: {settings.chunk_size}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Proyecto GCP: holmesgraphrag
Neo4j URI: bolt://localhost:7687
Chunk size: 1500


## 1. Inicializar Neo4j y crear esquema

In [16]:
neo4j = Neo4jManager()
neo4j.setup_database()  # Crea constraints + índices vectoriales + fulltext
print("Base de datos inicializada")
print(f"Stats actuales: {neo4j.get_stats()}")

Base de datos inicializada
Stats actuales: {}


## 2. Descargar y procesar textos de Gutenberg

In [17]:
processor = TextProcessor(neo4j_manager=neo4j)

#Solo los 10 relatos de desarrollo
story_chunks = processor.process_phase1()

print(f"\nRelatos procesados: {len(story_chunks)}")
for title, chunks in story_chunks.items():
    print(f"  - {title}: {len(chunks)} chunks")

Generando embeddings: 100%|██████████| 35/35 [00:27<00:00,  1.27it/s]                          
                                                                                            


Relatos procesados: 10
  - Silver Blaze: 36 chunks
  - The Final Problem: 27 chunks
  - A Scandal In Bohemia: 32 chunks
  - The Red-Headed League: 34 chunks
  - A Case Of Identity: 26 chunks
  - The Five Orange Pips: 27 chunks
  - The Adventure Of The Blue Carbuncle: 29 chunks
  - The Adventure Of The Speckled Band: 67 chunks
  - The Adventure Of The Copper Beeches: 37 chunks
  - The Adventure Of The Dancing Men: 35 chunks


## 3. Extracción de entidades y relaciones

Extracción multipaso con sliding context:
- **Paso 1**: Extracción de entidades (Characters, Locations, Crimes, Objects, Deductions, Scenes, Events)
- **Paso 2**: Extracción de relaciones entre las entidades encontradas
- **Entity Resolution**: Normalización + embeddings + LLM para desambiguar duplicados

In [18]:
'''
import logging
logging.getLogger('graphrag.ingestion.entity_extractor').setLevel(logging.DEBUG)

extractor = EntityExtractor()

all_results = {}
story_title = "A SCANDAL IN BOHEMIA"
chunks = story_chunks[story_title]

# Solo 3 chunks para debug rápido
result = extractor.process_story_chunks(chunks, story_title)
all_results[story_title] = result

entities = result["entities"]
print(f"Personajes: {len(entities.get('characters', []))}")
for c in entities["characters"]:
  print(f"  [{c['name']}] aliases: {c.get('aliases', [])}")
'''

'\nimport logging\nlogging.getLogger(\'graphrag.ingestion.entity_extractor\').setLevel(logging.DEBUG)\n\nextractor = EntityExtractor()\n\nall_results = {}\nstory_title = "A SCANDAL IN BOHEMIA"\nchunks = story_chunks[story_title]\n\n# Solo 3 chunks para debug rápido\nresult = extractor.process_story_chunks(chunks, story_title)\nall_results[story_title] = result\n\nentities = result["entities"]\nprint(f"Personajes: {len(entities.get(\'characters\', []))}")\nfor c in entities["characters"]:\n  print(f"  [{c[\'name\']}] aliases: {c.get(\'aliases\', [])}")\n'

In [19]:
extractor = EntityExtractor()

# Para prueba — quitar el slice para el run completo
#TEST_STORIES = ["The Final Problem", "A Case Of Identity", "The Red-Headed League"]

all_results = {}
for story_title, chunks in story_chunks.items():
#    if story_title not in TEST_STORIES:
#        continue

    print(f"\n{'='*60}")
    print(f"Procesando: {story_title}")
    print(f"{'='*60}")

    result = extractor.process_story_chunks(chunks, story_title)
    all_results[story_title] = result

    # Resumen
    entities = result["entities"]
    n_chars = len(entities.get("characters", []))
    n_locs = len(entities.get("locations", []))
    n_crimes = len(entities.get("crimes", []))
    n_deductions = len(entities.get("deductions", []))
    print(f"  Personajes: {n_chars}, Ubicaciones: {n_locs}, Crímenes: {n_crimes}, Deducciones: {n_deductions}")
    print(f"  Relaciones: {len(result['relationships'])}")


Procesando: Silver Blaze


Generando embeddings: 100%|██████████| 100/100 [00:05<00:00, 18.02it/s]


  Personajes: 28, Ubicaciones: 26, Crímenes: 8, Deducciones: 33
  Relaciones: 630

Procesando: The Final Problem


Generando embeddings: 100%|██████████| 69/69 [00:04<00:00, 15.57it/s]


  Personajes: 12, Ubicaciones: 17, Crímenes: 7, Deducciones: 16
  Relaciones: 349

Procesando: A Scandal In Bohemia


...

  Personajes: 40, Ubicaciones: 53, Crímenes: 21, Deducciones: 36
  Relaciones: 738

Procesando: The Adventure Of The Copper Beeches


Generando embeddings: 100%|██████████| 107/107 [00:05<00:00, 18.63it/s]


  Personajes: 19, Ubicaciones: 37, Crímenes: 5, Deducciones: 18
  Relaciones: 528

Procesando: The Adventure Of The Dancing Men


Extrayendo 'The Adventure Of The Dancing Men':  83%|████████▎ | 29/35 [15:57<03:28, 34.69s/it]Error extrayendo entidades: [Errno 11001] getaddrinfo failed. Devolviendo resultado vacío.
Chunk 30/35 de 'The Adventure Of The Dancing Men' devolvió resultado vacío.
Error extrayendo relaciones: [Errno 11001] getaddrinfo failed. Devolviendo resultado vacío.
Error extrayendo entidades: [Errno 11001] getaddrinfo failed. Devolviendo resultado vacío.
Chunk 31/35 de 'The Adventure Of The Dancing Men' devolvió resultado vacío.
Error extrayendo relaciones: [Errno 11001] getaddrinfo failed. Devolviendo resultado vacío.
Error extrayendo entidades: [Errno 11001] getaddrinfo failed. Devolviendo resultado vacío.
Chunk 32/35 de 'The Adventure Of The Dancing Men' devolvió resultado vacío.
Error extrayendo relaciones: [Errno 11001] getaddrinfo failed. Devolviendo resultado vacío.
Error extrayendo entidades: [Errno 11001] getaddrinfo failed. Devolviendo resultado vacío.
Chunk 33/35 de 'The Adventure Of The D

  Personajes: 19, Ubicaciones: 26, Crímenes: 11, Deducciones: 31
  Relaciones: 513


In [20]:
import json
import os

# Guarda los resultados de extracción a disco por si el pipeline se interrumpe.
# Para recargar sin re-extraer: all_results = json.load(open("../output/extraction_results.json"))
os.makedirs("../output", exist_ok=True)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"Checkpoint guardado: ../output/extraction_results.json ({len(all_results)} relatos)")

Checkpoint guardado: ../output/extraction_results.json (10 relatos)


In [21]:
# Resolución cross-story: unifica nombres canónicos entre relatos.
# Garantiza que Holmes y Watson tengan el mismo nombre canónico en Neo4j
# independientemente del relato de origen.
all_results = extractor.normalize_cross_story_entities(all_results)

# Guarda el checkpoint normalizado (sobreescribe el anterior)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("Cross-story normalization completada.")
# Verificación rápida
for story, result in all_results.items():
    chars = result["entities"].get("characters", [])
    holmes = next((c["name"] for c in chars if "holmes" in c["name"].lower()), "--")
    watson = next((c["name"] for c in chars if "watson" in c["name"].lower()), "--")
    print(f"  {story[:40]:40s}  Holmes='{holmes}'  Watson='{watson}'")

Cross-story normalization completada.
  Silver Blaze                              Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Final Problem                         Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  A Scandal In Bohemia                      Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Red-Headed League                     Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  A Case Of Identity                        Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Five Orange Pips                      Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Blue Carbuncle       Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Speckled Band        Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Copper Beeches       Holmes='Sherlock Holmes'  Watson='Dr. Watson'
  The Adventure Of The Dancing Men          Holmes='Sherlock Holmes'  Watson='Dr. Watson'


## 4. Poblar el grafo en Neo4j

In [22]:
for story_title, result in all_results.items():
    print(f"Almacenando: {story_title}")

    # Almacenar entidades
    neo4j.store_entities(result["entities"], story_title)

    # Almacenar relaciones
    neo4j.store_relationships(result["relationships"], story_title=story_title)

    # Vincular chunks con las entidades que mencionan
    for chunk_info in result.get("chunk_entities", []):
        if chunk_info["chunk_id"] and chunk_info["entity_names"]:
            neo4j.link_chunk_to_entities(chunk_info["chunk_id"], chunk_info["entity_names"])

print("\nGrafo poblado exitosamente")

Almacenando: Silver Blaze
Almacenando: The Final Problem
Almacenando: A Scandal In Bohemia
Almacenando: The Red-Headed League
Almacenando: A Case Of Identity
Almacenando: The Five Orange Pips
Almacenando: The Adventure Of The Blue Carbuncle
Almacenando: The Adventure Of The Speckled Band
Almacenando: The Adventure Of The Copper Beeches
Almacenando: The Adventure Of The Dancing Men

Grafo poblado exitosamente


## 5. Verificar el grafo

In [23]:
stats = neo4j.get_stats()
print("Estadísticas del grafo:")
for label, count in stats.items():
    print(f"  {label}: {count}")

Estadísticas del grafo:
  Event: 592
  Object: 521
  Chunk: 350
  Scene: 279
  Deduction: 262
  Location: 218
  Character: 189
  Crime: 86
  Story: 10


In [24]:
# Ver personajes más conectados
top_characters = neo4j.execute_query("""
MATCH (c:Character)-[r]-()
RETURN c.name AS name, count(DISTINCT r) AS connections
ORDER BY connections DESC
LIMIT 10
""")

print("\nPersonajes más conectados:")
for char in top_characters:
    print(f"  {char['name']}: {char['connections']} conexiones")


Personajes más conectados:
  Sherlock Holmes: 786 conexiones
  Dr. Watson: 303 conexiones
  Mr. Duncan Ross: 60 conexiones
  John Straker: 59 conexiones
  Mr. Jabez Wilson: 57 conexiones
  VIOLET HUNTER: 55 conexiones
  Dr. Grimesby Roylott: 53 conexiones
  Miss Alice Rucastle: 47 conexiones
  Mr. Hilton Cubitt: 45 conexiones
  Mr. Hosmer Angel: 43 conexiones


In [25]:
# Ver relatos y sus entidades
stories = neo4j.execute_query("""
MATCH (s:Story)
OPTIONAL MATCH (c:Character)-[:APPEARS_IN]->(s)
RETURN s.title AS story, s.collection AS collection, count(c) AS characters
ORDER BY characters DESC
""")

print("\nRelatos cargados:")
for s in stories:
    print(f"  {s['story']} ({s['collection']}): {s['characters']} personajes")


Relatos cargados:
  The Adventure Of The Speckled Band (The Adventures of Sherlock Holmes): 40 personajes
  The Adventure Of The Blue Carbuncle (The Adventures of Sherlock Holmes): 32 personajes
  Silver Blaze (The Memoirs of Sherlock Holmes): 28 personajes
  A Scandal In Bohemia (The Adventures of Sherlock Holmes): 23 personajes
  The Five Orange Pips (The Adventures of Sherlock Holmes): 21 personajes
  The Red-Headed League (The Adventures of Sherlock Holmes): 20 personajes
  A Case Of Identity (The Adventures of Sherlock Holmes): 19 personajes
  The Adventure Of The Copper Beeches (The Adventures of Sherlock Holmes): 19 personajes
  The Adventure Of The Dancing Men (The Return of Sherlock Holmes): 19 personajes
  The Final Problem (The Memoirs of Sherlock Holmes): 12 personajes


In [26]:
# Ver cadenas de deducción
deductions = neo4j.execute_query("""
MATCH (d:Deduction)-[:LEADS_TO]->(d2:Deduction)
RETURN d.observation AS from_obs, d2.observation AS to_obs
LIMIT 5
""")

if deductions:
    print("\nCadenas de deducción encontradas:")
    for d in deductions:
        print(f"  {d['from_obs'][:60]}... → {d['to_obs'][:60]}...")
else:
    print("\nNo se encontraron cadenas de deducción (LEADS_TO)")


Cadenas de deducción encontradas:
  the silence of the dog... → John Straker went down to the stables in the dead of the nig...
  A large 'E' with a small 'g,' a 'P,' and a large 'G' with a ... → The initials 'Eg' on the paper, combined with a search in th...
  Holmes falls upon his knees upon the floor and, with the lan... → Holmes has completed his examination of the floor and puts h...
  The assistant having come for half wages... → The man’s business was a small one, and there was nothing in...
  The man’s business was a small one, and there was nothing in... → I thought of the assistant’s fondness for photography, and h...


## 6. Cleanup (opcional)

In [27]:
 # Descomentar para limpiar la base de datos completa
#neo4j.clear_database()
#print("Base de datos limpiada")

neo4j.close()
print("Conexión cerrada")

Conexión cerrada
